# Fashion Retail Demand Forecasting

**Research question:** Do Google Trends signals for fashion search terms improve forecasts of monthly US clothing retail sales?

This notebook walks through the full analysis:
1. Data loading and exploration
2. Preprocessing (inflation adjustment, stationarity testing)
3. Model fitting and comparison (5 model variants)
4. Rolling-window cross-validation
5. Key findings

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

from src.config import (
    ANALYSIS_START, DATA_RAW, FIGURES_DIR, GOOGLE_TRENDS_DIR, RESULTS_DIR, TRENDS_TERMS,
)
from src.data_loader import fetch_clothing_sales, fetch_cpi_apparel, load_google_trends, save_raw_data
from src.preprocessing import add_time_features, check_stationarity, deflate_sales, merge_datasets
from src.models import SeasonalNaive, SARIMAXModel, ProphetModel
from src.evaluation import compute_all_metrics, rolling_window_cv

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 120

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data Loading

In [ ]:
# Fetch FRED data (requires FRED_API_KEY in .env)
clothing_sales = fetch_clothing_sales()
cpi_apparel = fetch_cpi_apparel()
save_raw_data(clothing_sales, cpi_apparel)

# Load Google Trends (manually exported CSVs)
trends = load_google_trends()

print(f"Clothing sales: {clothing_sales.index[0]} to {clothing_sales.index[-1]} ({len(clothing_sales)} months)")
print(f"CPI Apparel:    {cpi_apparel.index[0]} to {cpi_apparel.index[-1]} ({len(cpi_apparel)} months)")
print(f"Google Trends:  {trends.index[0]} to {trends.index[-1]} ({len(trends)} months)")

## 2. Data Exploration

In [ ]:
# Raw clothing retail sales time series
fig, ax = plt.subplots(figsize=(14, 5))
ts_index = clothing_sales.index.to_timestamp()
ax.plot(ts_index, clothing_sales.values, linewidth=0.8)
ax.set_title("US Monthly Clothing Retail Sales (Nominal)", fontsize=14)
ax.set_ylabel("Millions USD")
ax.set_xlabel("")

# Annotate recessions and COVID
ax.axvspan(pd.Timestamp("2007-12"), pd.Timestamp("2009-06"), alpha=0.15, color="gray", label="Great Recession")
ax.axvspan(pd.Timestamp("2020-02"), pd.Timestamp("2020-04"), alpha=0.15, color="red", label="COVID-19")
ax.legend(loc="upper left")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "raw_sales_timeseries.png", bbox_inches="tight")
plt.show()

In [ ]:
# Seasonal decomposition (multiplicative, since variance grows with level)
decomp = seasonal_decompose(clothing_sales.dropna(), model="multiplicative", period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title="Observed")
decomp.trend.plot(ax=axes[1], title="Trend")
decomp.seasonal.plot(ax=axes[2], title="Seasonal")
decomp.resid.plot(ax=axes[3], title="Residual")
for ax in axes:
    ax.set_xlabel("")
plt.suptitle("Seasonal Decomposition (Multiplicative)", fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "seasonal_decomposition.png", bbox_inches="tight")
plt.show()

In [ ]:
# Google Trends over time
fig, ax = plt.subplots(figsize=(14, 5))
for col in trends.columns:
    ax.plot(trends.index.to_timestamp(), trends[col].values, label=col.replace("_", " ").title(), linewidth=0.8)
ax.set_title("Google Trends: Fashion Search Interest Over Time", fontsize=14)
ax.set_ylabel("Relative Search Interest (0-100)")
ax.legend(loc="upper left")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "google_trends_timeseries.png", bbox_inches="tight")
plt.show()

## 3. Preprocessing

In [ ]:
# Merge all sources and deflate to real dollars
df = merge_datasets(clothing_sales, cpi_apparel, trends=trends)
df = add_time_features(df)

print(f"Analysis dataset: {df.index[0]} to {df.index[-1]} ({len(df)} months)")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Compare nominal vs. real (inflation-adjusted) sales
fig, ax = plt.subplots(figsize=(14, 5))
ts_idx = df.index.to_timestamp()
ax.plot(ts_idx, df["clothing_sales"].values, label="Nominal", linewidth=0.8)
ax.plot(ts_idx, df["real_clothing_sales"].values, label="Real (CPI-adjusted)", linewidth=0.8)
ax.set_title("Nominal vs. Real Clothing Retail Sales", fontsize=14)
ax.set_ylabel("Millions USD")
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / "nominal_vs_real_sales.png", bbox_inches="tight")
plt.show()

In [ ]:
# Stationarity tests
target = df["real_clothing_sales"]

print("ADF Test on Real Clothing Sales (levels):")
adf_levels = check_stationarity(target)
print(f"  Test statistic: {adf_levels['test_statistic']:.4f}")
print(f"  p-value:        {adf_levels['p_value']:.4f}")
print(f"  Stationary:     {adf_levels['is_stationary']}")

print("\nADF Test on First Difference:")
adf_diff = check_stationarity(target.diff().dropna())
print(f"  Test statistic: {adf_diff['test_statistic']:.4f}")
print(f"  p-value:        {adf_diff['p_value']:.4f}")
print(f"  Stationary:     {adf_diff['is_stationary']}")

In [ ]:
# Correlation heatmap: Trends terms vs. real sales
trends_cols = [c for c in df.columns if c in trends.columns]
corr_cols = ["real_clothing_sales"] + trends_cols
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlation: Real Sales vs. Google Trends", fontsize=14)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "correlation_heatmap.png", bbox_inches="tight")
plt.show()

## 4. Train/Test Split

In [ ]:
# Use the last 24 months as a holdout test set for visualization
# (Cross-validation in the next section is the primary evaluation method)
HOLDOUT_MONTHS = 24

y = df["real_clothing_sales"]
exog = df[trends_cols]

train_y = y.iloc[:-HOLDOUT_MONTHS]
test_y = y.iloc[-HOLDOUT_MONTHS:]
train_exog = exog.iloc[:-HOLDOUT_MONTHS]
test_exog = exog.iloc[-HOLDOUT_MONTHS:]

print(f"Training: {train_y.index[0]} to {train_y.index[-1]} ({len(train_y)} months)")
print(f"Test:     {test_y.index[0]} to {test_y.index[-1]} ({len(test_y)} months)")

# Visualize the split
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_y.index.to_timestamp(), train_y.values, label="Train", linewidth=0.8)
ax.plot(test_y.index.to_timestamp(), test_y.values, label="Test", linewidth=0.8, color="darkorange")
ax.axvline(test_y.index[0].to_timestamp(), color="black", linestyle="--", linewidth=0.8, label="Split")
ax.set_title("Train/Test Split", fontsize=14)
ax.set_ylabel("Real Sales (Millions USD)")
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / "train_test_split.png", bbox_inches="tight")
plt.show()

## 5. Model Fitting and Forecast Comparison

In [ ]:
# Fit all 5 model variants on the training set and forecast the test period
models = {
    "Seasonal Naive": (SeasonalNaive, {"use_exog": False}),
    "SARIMAX": (SARIMAXModel, {"use_exog": False}),
    "SARIMAX + Trends": (SARIMAXModel, {"use_exog": True}),
    "Prophet": (ProphetModel, {"use_exog": False}),
    "Prophet + Trends": (ProphetModel, {"use_exog": True}),
}

forecasts = {}
holdout_metrics = {}

for name, (model_cls, opts) in models.items():
    print(f"Fitting {name}...")
    model = model_cls()
    
    fit_exog = train_exog if opts["use_exog"] else None
    pred_exog = test_exog if opts["use_exog"] else None
    
    model.fit(train_y, exog=fit_exog)
    preds = model.predict(steps=HOLDOUT_MONTHS, exog=pred_exog)
    
    forecasts[name] = preds
    holdout_metrics[name] = compute_all_metrics(test_y.values, preds)
    print(f"  RMSE: {holdout_metrics[name]['rmse']:.1f}  MAE: {holdout_metrics[name]['mae']:.1f}  MAPE: {holdout_metrics[name]['mape']:.1f}%")

print("\nDone.")

In [ ]:
# Overlay all forecasts vs. actual
fig, ax = plt.subplots(figsize=(14, 6))

# Plot actual (train tail + test)
tail_months = 48
plot_train = train_y.iloc[-tail_months:]
ax.plot(plot_train.index.to_timestamp(), plot_train.values, color="black", linewidth=1, label="Actual (train)")
ax.plot(test_y.index.to_timestamp(), test_y.values, color="black", linewidth=1.5, linestyle="--", label="Actual (test)")

# Plot each model's forecast
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
for (name, preds), color in zip(forecasts.items(), colors):
    ax.plot(test_y.index.to_timestamp(), preds, label=name, linewidth=1, color=color, alpha=0.8)

ax.axvline(test_y.index[0].to_timestamp(), color="gray", linestyle=":", linewidth=0.8)
ax.set_title("Forecast Comparison: All Models vs. Actual", fontsize=14)
ax.set_ylabel("Real Sales (Millions USD)")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "forecast_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
# Holdout metrics table
metrics_df = pd.DataFrame(holdout_metrics).T
metrics_df = metrics_df.round(2)
metrics_df.columns = ["RMSE", "MAE", "MAPE (%)"]
metrics_df = metrics_df.sort_values("RMSE")
print("Holdout Set Metrics (last 24 months):")
metrics_df

## 6. Rolling-Window Cross-Validation

In [ ]:
# Run rolling-window CV for all 5 model variants
# This is the primary evaluation: more robust than a single train/test split

cv_results = {}

model_configs = {
    "Seasonal Naive": (SeasonalNaive, False),
    "SARIMAX": (SARIMAXModel, False),
    "SARIMAX + Trends": (SARIMAXModel, True),
    "Prophet": (ProphetModel, False),
    "Prophet + Trends": (ProphetModel, True),
}

for name, (model_cls, use_exog) in model_configs.items():
    print(f"Cross-validating {name}...")
    results = rolling_window_cv(
        model_factory=model_cls,
        y=y,
        exog=exog if use_exog else None,
    )
    cv_results[name] = results
    mean_rmse = np.mean([r["metrics"]["rmse"] for r in results])
    print(f"  {len(results)} folds, mean RMSE: {mean_rmse:.1f}")

print("\nDone.")

In [ ]:
# Aggregate CV metrics
cv_summary = []
for name, results in cv_results.items():
    rmses = [r["metrics"]["rmse"] for r in results]
    maes = [r["metrics"]["mae"] for r in results]
    mapes = [r["metrics"]["mape"] for r in results]
    cv_summary.append({
        "Model": name,
        "Mean RMSE": np.mean(rmses),
        "Std RMSE": np.std(rmses),
        "Mean MAE": np.mean(maes),
        "Mean MAPE (%)": np.mean(mapes),
        "Folds": len(results),
    })

cv_df = pd.DataFrame(cv_summary).set_index("Model").sort_values("Mean RMSE")
cv_df = cv_df.round(2)
print("Cross-Validation Results (sorted by Mean RMSE):")
cv_df

In [ ]:
# Bar chart of mean CV metrics by model
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

model_order = cv_df.index.tolist()
colors = sns.color_palette("muted", len(model_order))

for ax, metric, label in zip(
    axes,
    ["Mean RMSE", "Mean MAE", "Mean MAPE (%)"],
    ["RMSE", "MAE", "MAPE (%)"]
):
    bars = ax.barh(model_order, cv_df[metric], color=colors)
    ax.set_xlabel(label)
    ax.invert_yaxis()

plt.suptitle("Cross-Validation: Mean Metrics by Model", fontsize=14)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "cv_metrics_bar_chart.png", bbox_inches="tight")
plt.show()

In [ ]:
# Per-fold RMSE line plot (shows stability across time)
fig, ax = plt.subplots(figsize=(14, 5))

for name, results in cv_results.items():
    fold_rmses = [r["metrics"]["rmse"] for r in results]
    fold_labels = [r["test_start"] for r in results]
    ax.plot(fold_labels, fold_rmses, marker="o", linewidth=1, markersize=5, label=name)

ax.set_title("Per-Fold RMSE Across Time", fontsize=14)
ax.set_ylabel("RMSE")
ax.set_xlabel("Test Fold Start")
ax.legend(loc="upper left", fontsize=9)
plt.xticks(rotation=45)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "cv_per_fold_rmse.png", bbox_inches="tight")
plt.show()

## 7. COVID Robustness

The COVID-19 pandemic caused an unprecedented crash in clothing retail sales (March-April 2020). How did each model handle this shock?

In [ ]:
# Identify folds that overlap with COVID (2020)
covid_folds = {}
for name, results in cv_results.items():
    for r in results:
        if "2020" in r["test_start"] or "2020" in r["test_end"]:
            covid_folds.setdefault(name, []).append(r)

if covid_folds:
    print("COVID-overlapping fold metrics:")
    for name, folds in covid_folds.items():
        for f in folds:
            print(f"  {name} | fold {f['fold']} ({f['test_start']} to {f['test_end']}): RMSE={f['metrics']['rmse']:.1f}")
else:
    print("No CV folds overlap with COVID period (depends on data length and CV parameters).")

In [ ]:
# If we have COVID folds, visualize actual vs. predicted for that period
if covid_folds:
    fig, axes = plt.subplots(len(covid_folds), 1, figsize=(14, 4 * len(covid_folds)), sharex=True)
    if len(covid_folds) == 1:
        axes = [axes]
    
    for ax, (name, folds) in zip(axes, covid_folds.items()):
        for f in folds:
            months = pd.period_range(f["test_start"], periods=len(f["actual"]), freq="M")
            ax.plot(months.to_timestamp(), f["actual"], "k--", linewidth=1.5, label="Actual")
            ax.plot(months.to_timestamp(), f["predicted"], linewidth=1, label=f"Predicted (RMSE={f['metrics']['rmse']:.0f})")
        ax.set_title(name, fontsize=12)
        ax.set_ylabel("Real Sales")
        ax.legend(fontsize=9)
    
    plt.suptitle("COVID Period: Actual vs. Predicted", fontsize=14)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "covid_robustness.png", bbox_inches="tight")
    plt.show()

## 8. Key Findings

In [ ]:
# Summary comparison: with vs. without Google Trends
print("=" * 60)
print("RESEARCH QUESTION: Do Google Trends signals improve forecasts?")
print("=" * 60)

pairs = [
    ("SARIMAX", "SARIMAX + Trends"),
    ("Prophet", "Prophet + Trends"),
]

for base, enhanced in pairs:
    base_rmse = cv_df.loc[base, "Mean RMSE"]
    enh_rmse = cv_df.loc[enhanced, "Mean RMSE"]
    pct_change = (enh_rmse - base_rmse) / base_rmse * 100
    direction = "improvement" if pct_change < 0 else "degradation"
    print(f"\n{base} vs. {enhanced}:")
    print(f"  {base:20s}  Mean RMSE = {base_rmse:.2f}")
    print(f"  {enhanced:20s}  Mean RMSE = {enh_rmse:.2f}")
    print(f"  Change: {pct_change:+.1f}% ({direction})")

print("\n" + "=" * 60)
best_model = cv_df.index[0]
print(f"Best overall model (by mean CV RMSE): {best_model}")
print(f"  Mean RMSE: {cv_df.loc[best_model, 'Mean RMSE']:.2f}")
print(f"  Mean MAPE: {cv_df.loc[best_model, 'Mean MAPE (%)']:.2f}%")

In [ ]:
# Save all metrics to JSON for the README
output = {
    "holdout_metrics": {name: {k: round(v, 2) for k, v in m.items()} for name, m in holdout_metrics.items()},
    "cv_summary": cv_df.reset_index().to_dict(orient="records"),
}

with open(RESULTS_DIR / "metrics.json", "w") as f:
    json.dump(output, f, indent=2)

print(f"Metrics saved to {RESULTS_DIR / 'metrics.json'}")